## 1. Introduction

The Poisson Generalized Linear Model assumes equidispersion, where the conditional mean and variance of the response variable are equal.

However, empirical evidence from the previous analysis indicates significant overdispersion in the claim frequency data. This violates the Poisson assumption and leads to underestimated standard errors and potentially biased inference.

To address this limitation, the Negative Binomial GLM is introduced. This model extends the Poisson framework by incorporating an additional dispersion parameter, allowing the variance to exceed the mean.

## 2. Statistical Motivation

Under the Negative Binomial specification:

\[
Var(Y_i) = \mu_i + \alpha \mu_i^2
\]

where:

- \( \mu_i \) is the expected claim frequency  
- \( \alpha \) is the dispersion parameter  

Unlike the Poisson model, which enforces \( Var(Y) = E(Y) \), the Negative Binomial model allows for unobserved heterogeneity across policyholders.

This makes it particularly suitable for insurance datasets where risk heterogeneity is not fully captured by observed covariates.

## 3. Model Specification

The Negative Binomial GLM is specified as:

\[
\log(\mu_i) = X_i \beta + \log(\text{Exposure}_i)
\]

\[
Y_i \sim NB(\mu_i, \alpha)
\]

where:

- \(X_i\): explanatory variables  
- \(\beta\): regression coefficients  
- \(\alpha\): dispersion parameter  
- Exposure is included as a log-offset to normalize claim counts over time.

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv(r"D:\Insurance_Pricing_Project\Data\freMTPL2freq.csv")

In [4]:
df.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region
0,1.0,1,0.10,D,5,0,55,50,B12,Regular,1217,R82
1,3.0,1,0.77,D,5,0,55,50,B12,Regular,1217,R82
2,5.0,1,0.75,B,6,2,52,50,B12,Diesel,54,R22
3,10.0,1,0.09,B,7,0,46,50,B12,Diesel,76,R72
4,11.0,1,0.84,B,7,0,46,50,B12,Diesel,76,R72


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 678013 entries, 0 to 678012
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   IDpol       678013 non-null  float64
 1   ClaimNb     678013 non-null  int64  
 2   Exposure    678013 non-null  float64
 3   Area        678013 non-null  object 
 4   VehPower    678013 non-null  int64  
 5   VehAge      678013 non-null  int64  
 6   DrivAge     678013 non-null  int64  
 7   BonusMalus  678013 non-null  int64  
 8   VehBrand    678013 non-null  object 
 9   VehGas      678013 non-null  object 
 10  Density     678013 non-null  int64  
 11  Region      678013 non-null  object 
dtypes: float64(2), int64(6), object(4)
memory usage: 62.1+ MB


In [4]:
df.columns

Index(['IDpol', 'ClaimNb', 'Exposure', 'Area', 'VehPower', 'VehAge', 'DrivAge',
       'BonusMalus', 'VehBrand', 'VehGas', 'Density', 'Region'],
      dtype='object')

## 4. Modeling Dataset Preparation

The modeling dataset is constructed by selecting relevant variables required for frequency modeling.

This step ensures consistency across different model specifications (Poisson and Negative Binomial GLMs) and enables fair comparison of results.

Only variables with known actuarial relevance are included in the baseline model.

In [6]:
model_df = df[
    [
        "ClaimNb",
        "Exposure",
        "Area",
        "VehPower",
        "VehAge",
        "DrivAge",
        "BonusMalus",
        "VehBrand",
        "VehGas",
        "Region",
        "Density"
    ]
].copy()

## 5. Train-Test Split

The dataset is split into training and testing subsets to evaluate model generalization performance.

The training set is used for parameter estimation, while the test set is reserved for out-of-sample evaluation.

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    random_state=42
)

print(train_df.shape, test_df.shape)

In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    random_state=42
)

print(train_df.shape)
print(test_df.shape)

(542410, 11)
(135603, 11)


## 6. Negative Binomial Model Specification

The Negative Binomial GLM extends the Poisson model by introducing a dispersion parameter, allowing the variance to exceed the mean.

The model is defined as:

\[
Y_i \sim NB(\mu_i, \alpha)
\]

\[
\log(\mu_i) = X_i \beta + \log(\text{Exposure}_i)
\]

This formulation allows the model to capture unobserved heterogeneity in claim frequency data.

In [7]:
formula = """
ClaimNb ~ Area + VehPower + VehAge + DrivAge 
+ BonusMalus + VehBrand + VehGas + Region + Density
"""

In [10]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

formula = """
ClaimNb ~ Area + VehPower + VehAge + DrivAge
        + BonusMalus + VehBrand + VehGas + Region + Density
"""

nb_model = smf.glm(
    formula=formula,
    data=train_df,
    family=sm.families.NegativeBinomial(alpha=1.0),
    offset=np.log(train_df["Exposure"])
).fit()

print(nb_model.summary())

MemoryError: 

## Note on Negative Binomial Dispersion

The Negative Binomial model includes a dispersion parameter (α) that controls the level of overdispersion.

In this implementation, dispersion is handled using Pearson scaling to ensure more robust variance estimation and avoid reliance on default parameter assumptions.

## 8. Model Comparison: Poisson vs Negative Binomial

To evaluate the impact of relaxing the equidispersion assumption, the Poisson and Negative Binomial models are compared using their coefficient estimates and predictive performance.

This comparison helps assess whether accounting for overdispersion leads to materially different risk assessments.

In [20]:
poisson_model = smf.glm(
    formula=formula,
    data=train_df,
    family=sm.families.Poisson(),
    offset=np.log(train_df["Exposure"])
).fit()

In [21]:
comparison = pd.DataFrame({
    "Poisson": poisson_model.params,
    "NegativeBinomial": nb_model.params
})

comparison

,Poisson,NegativeBinomial
Intercept,-4.000787e+00,-4.009703e+00
Area[T.B],5.917298e-02,5.928405e-02
Area[T.C],9.090674e-02,9.061261e-02
Area[T.D],1.874106e-01,1.896957e-01
Area[T.E],2.320516e-01,2.319616e-01
Area[T.F],2.324745e-01,2.357561e-01
VehBrand[T.B10],-4.035624e-02,-3.721197e-02
VehBrand[T.B11],2.808186e-02,2.958819e-02
VehBrand[T.B12],1.637002e-01,1.825996e-01
VehBrand[T.B13],5.690394e-03,2.645176e-03


## 9. Out-of-Sample Predictions

Predicted claim frequencies are generated for both models on the test dataset.

This allows direct comparison of model performance on unseen data.

In [22]:
test_df = test_df.copy()

test_df["Pred_Poisson"] = poisson_model.predict(
    test_df,
    offset=np.log(test_df["Exposure"])
)

test_df["Pred_NB"] = nb_model.predict(
    test_df,
    offset=np.log(test_df["Exposure"])
)

## 10. Model Evaluation

Model performance is evaluated using standard regression metrics:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Poisson Deviance

These metrics provide insight into predictive accuracy and goodness-of-fit.

In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_poisson_deviance
import numpy as np

def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Deviance": mean_poisson_deviance(y_true, y_pred)
    }

results = pd.DataFrame({
    "Poisson": evaluate(test_df["ClaimNb"], test_df["Pred_Poisson"]),
    "NegativeBinomial": evaluate(test_df["ClaimNb"], test_df["Pred_NB"])
})

results

,Poisson,NegativeBinomial
MAE,0.098892,0.099472
RMSE,0.237330,0.237424
Deviance,0.321607,0.321650


## 11. Final Model Comparison Insight

Although the Negative Binomial GLM was introduced to address overdispersion in the claim frequency data, the out-of-sample results show minimal improvement over the Poisson GLM.

This suggests that:

- The Poisson model is already a strong baseline for predictive performance
- Overdispersion primarily affects inference (standard errors), rather than prediction accuracy
- The additional flexibility of the Negative Binomial model does not materially improve predictive power in this dataset

In [2]:
import os
print(os.getcwd())

C:\Users\Umer


In [25]:
test_df = test_df.copy()

test_df["NB_Predicted"] = nb_model.predict(
    test_df,
    offset=np.log(test_df["Exposure"])
)

print(test_df[["ClaimNb", "NB_Predicted"]].head())

        ClaimNb  NB_Predicted
261354        0      0.065502
448143        0      0.019487
188618        0      0.059030
12952         0      0.054598
425028        0      0.031438


In [26]:
import os

save_folder = r"C:\Users\Umer\predictions"
os.makedirs(save_folder, exist_ok=True)

nb_output = test_df[[
    "ClaimNb",
    "NB_Predicted"
]].copy()

nb_output.rename(
    columns={
        "ClaimNb": "Actual_Frequency",
        "NB_Predicted": "NegativeBinomial_Frequency"
    },
    inplace=True
)

file_path = os.path.join(
    save_folder,
    "negative_binomial_frequency_predictions.csv"
)

nb_output.to_csv(
    file_path,
    index=False
)

print("Saved successfully:", file_path)

Saved successfully: C:\Users\Umer\predictions\negative_binomial_frequency_predictions.csv


In [15]:
print(train_df.shape)

(542410, 11)


In [16]:
print(formula)


ClaimNb ~ Area + VehPower + VehAge + DrivAge 
+ BonusMalus + VehBrand + VehGas + Region + Density



In [17]:
print(train_df.dtypes)

ClaimNb         int64
Exposure      float64
Area           object
VehPower        int64
VehAge          int64
DrivAge         int64
BonusMalus      int64
VehBrand       object
VehGas         object
Region         object
Density         int64
dtype: object


In [18]:
print(train_df.nunique())

ClaimNb         10
Exposure       184
Area             6
VehPower        12
VehAge          76
DrivAge         83
BonusMalus     112
VehBrand        11
VehGas           2
Region          22
Density       1607
dtype: int64


In [19]:
import psutil
print(psutil.virtual_memory())

svmem(total=8460189696, available=1270415360, percent=85.0, used=7189774336, free=1270415360)


In [11]:
import patsy

y, X = patsy.dmatrices(formula, train_df, return_type="dataframe")

print(X.shape)
print(f"Memory: {X.memory_usage(deep=True).sum()/1024**2:.2f} MB")

(542410, 43)
Memory: 182.08 MB
